# 1. What you'll learn

        - derive the same coefficients with a closed-form solve and gradient descent
- read residual plots instead of trusting RMSE alone
- recognize curvature and changing noise as assumption failures

        **The one question this notebook answers:** How does a straight-line model turn several measurements into an honest numerical prediction?

# 2. Setup

We keep the matrix algebra visible rather than hiding it behind one estimator call. The only optimizer state is a coefficient vector updated from a reproducible starting point.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We generate 700 rows from a known rule: target = 3 + 2·x₁ − 1.5·x₂ + 0.7·x₃ + noise. Noise grows with |x₁|, so the coefficients are recoverable but residual diagnostics reveal heteroscedasticity.

**Small example:** For x₁ values [0,1,2,3,4] and y=[1,3,5,7,9], each one-unit step adds 2. The slope is (9−1)/(4−0)=2 and the intercept is 1.

In [ ]:
rng = np.random.default_rng(SEED)
n = 700
X = rng.normal(size=(n, 3))
noise_sd = 0.45 + 0.35 * np.abs(X[:, 0])
noise = rng.normal(0, noise_sd)
true_beta = np.array([2.0, -1.5, 0.7])
y = 3.0 + X @ true_beta + noise
features = ["x1", "x2", "x3"]
frame = pd.DataFrame(X, columns=features)
frame["target"] = y
frame["noise_sd"] = noise_sd
print("Shape:", frame.shape)
print("True intercept/coefs:", 3.0, true_beta)
print(frame.head(3).round(3))

# 4. Explore

A linear model is plausible only when relationships are roughly straight, extreme values are understood, and predictors are not merely copies. These three panels inspect those facts before fitting.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].scatter(frame["x1"], frame["target"], s=12, alpha=.35)
axes[0].set(title="Target versus x1", xlabel="x1", ylabel="target")
sns.histplot(frame["target"], bins=25, kde=True, ax=axes[1], color="#176b66")
axes[1].set_title("Target distribution")
sns.heatmap(frame[features + ["target"]].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[2])
axes[2].set_title("Linear correlations")
plt.tight_layout(); plt.show()

**Takeaway:** The x₁ relationship is directional but noisy, which is appropriate for regression rather than a lookup table.

**Takeaway:** The target is continuous with no single dominant spike, so RMSE has a meaningful scale.

**Takeaway:** Each feature contributes distinct signal; no pair is an exact duplicate.

# 5. Prepare

We make 60/20/20 train, validation, and test partitions. The scaler is fitted on training only because gradient descent is sensitive to units; the closed form and gradient descent then solve the identical standardized problem.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
development, test = train_test_split(frame, test_size=.20, random_state=SEED)
train, validation = train_test_split(development, test_size=.25, random_state=SEED)
scaler = StandardScaler().fit(train[features])
X_train = scaler.transform(train[features]); X_val = scaler.transform(validation[features]); X_test = scaler.transform(test[features])
y_train = train["target"].to_numpy(); y_val = validation["target"].to_numpy(); y_test = test["target"].to_numpy()
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Training-only means:", scaler.mean_.round(3))

# 6. Train

        The closed form jumps directly to the least-squares minimum. Gradient descent approaches the same point in small steps. Seeing the two agree distinguishes an optimization method from the model it fits.

        Conceptually, fitting follows these steps:

        1. Add a column of ones so the intercept is a coefficient.
2. Closed form: multiply by the pseudo-inverse to minimize squared residuals.
3. Gradient descent: compute the prediction error, its slope with respect to each coefficient, and step downhill.
4. Stop after the loss has flattened and compare both coefficient vectors.

In [ ]:
X_design = np.column_stack([np.ones(len(X_train)), X_train])
beta_closed = np.linalg.pinv(X_design) @ y_train
beta_gd = np.zeros(X_design.shape[1]); learning_rate = .08; loss_history = []
for step in range(900):
    error = X_design @ beta_gd - y_train
    loss_history.append(np.mean(error ** 2))
    beta_gd -= learning_rate * (2 / len(X_design)) * X_design.T @ error
print("Closed form:", beta_closed.round(4))
print("Gradient descent:", beta_gd.round(4))
print("Maximum difference:", np.max(np.abs(beta_closed - beta_gd)).round(8))

# 7. Evaluate

RMSE expresses typical error in target units and R² compares against the mean baseline. Residuals should be centered without a systematic curve; their widening here correctly exposes the noise design.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
def predict(design_X, beta): return np.column_stack([np.ones(len(design_X)), design_X]) @ beta
pred = predict(X_test, beta_closed); baseline = np.repeat(y_train.mean(), len(y_test))
print(f"Model RMSE={mean_squared_error(y_test,pred)**.5:.3f}, R2={r2_score(y_test,pred):.3f}")
print(f"Mean baseline RMSE={mean_squared_error(y_test,baseline)**.5:.3f}")
residual = y_test - pred
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].scatter(pred, y_test, s=14, alpha=.5); axes[0].plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],"--",color="#d38b45"); axes[0].set(title="Predicted versus actual",xlabel="predicted",ylabel="actual")
axes[1].scatter(pred, residual, s=14, alpha=.5); axes[1].axhline(0,ls="--",color="#d38b45"); axes[1].set(title="Residual diagnostic",xlabel="predicted",ylabel="residual")
stats.probplot(residual, dist="norm", plot=axes[2]); axes[2].set_title("Residual Q–Q plot")
plt.tight_layout(); plt.savefig(Path.cwd()/"key_figure.png",dpi=160,bbox_inches="tight"); plt.show()

**Takeaway:** Both solvers recover the same linear rule and beat the mean, while the fan-shaped residual spread warns that one constant error bar would be misleading.

# 8. Break it

We add a strong x₁² term but insist on the original straight-plane model. Training still returns coefficients; the residual pattern and RMSE show that successful optimization cannot repair a misspecified hypothesis class.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
curved_y = y + 4 * X[:,0]**2
broken_train, broken_test = train_test_split(pd.DataFrame(X,columns=features).assign(target=curved_y),test_size=.25,random_state=SEED)
linear = LinearRegression().fit(broken_train[features],broken_train.target)
with_curve = LinearRegression().fit(np.column_stack([broken_train[features],broken_train.x1**2]),broken_train.target)
rmse_linear = mean_squared_error(broken_test.target,linear.predict(broken_test[features]))**.5
rmse_curve = mean_squared_error(broken_test.target,with_curve.predict(np.column_stack([broken_test[features],broken_test.x1**2])))**.5
print(f"Misspecified linear RMSE={rmse_linear:.3f}")
print(f"Add known curvature RMSE={rmse_curve:.3f}")
print(f"Damage from forcing a line={rmse_linear-rmse_curve:.3f}")

**Use this when…** a continuous target changes approximately additively with measured inputs and coefficient transparency matters.

        **Don't use this when…** residuals show strong curvature, changing variance, dependence, or extreme outliers that the data process—not merely a transformation—must address.

        ## Try this

        1. Double the noise standard deviation and watch coefficients remain roughly right while RMSE rises.
2. Remove scaling and lower the gradient-descent learning rate; convergence becomes unit-dependent.
3. Add x₁² as a feature to the healthy data; validation should reject unnecessary curvature.